# 03 — Noisy Evaluation
Zero-shot robustness: evaluate baseline models on OCR/ASR/Social noisy test sets.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

import numpy as np
import pandas as pd
import torch
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    BertForTokenClassification,
    GPT2ForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
from train import (
    ByT5ForTokenClassification,
    tokenize_and_align_labels,
    make_compute_metrics,
)

In [ ]:
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(f"Device: {DEVICE}")

label_names = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
id2label = {i: l for i, l in enumerate(label_names)}
label2id = {l: i for i, l in enumerate(label_names)}
num_labels = len(label_names)

RESULTS_DIR = os.path.join(os.path.dirname(os.getcwd()), "results")
CHECKPOINTS_DIR = os.path.join(RESULTS_DIR, "checkpoints")
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data", "noisy")

In [ ]:
def tokenize_and_align_labels_byt5(examples, tokenizer, max_length=256):
    all_input_ids, all_attention_masks, all_labels = [], [], []
    for tokens, ner_tags in zip(examples["tokens"], examples["ner_tags"]):
        input_ids = [0]
        label_ids = [-100]
        for word, label in zip(tokens, ner_tags):
            word_ids = tokenizer(word, add_special_tokens=False)["input_ids"]
            input_ids += word_ids
            label_ids += [label] + [-100] * (len(word_ids) - 1)
        input_ids = input_ids[:max_length]
        label_ids = label_ids[:max_length]
        pad_len = max_length - len(input_ids)
        all_input_ids.append(input_ids + [0] * pad_len)
        all_attention_masks.append([1]*len(input_ids) + [0]*pad_len)
        all_labels.append(label_ids + [-100]*pad_len)
    return {"input_ids": all_input_ids, "attention_mask": all_attention_masks, "labels": all_labels}

## Load pre-trained checkpoints

In [ ]:
import glob

def find_best_checkpoint(model_key):
    """Find the checkpoint with highest step number in results/checkpoints/{model_key}/"""
    model_dir = os.path.join(CHECKPOINTS_DIR, model_key)
    checkpoints = sorted(glob.glob(os.path.join(model_dir, "checkpoint-*")))
    if not checkpoints:
        raise FileNotFoundError(f"No checkpoints found in {model_dir}")
    return checkpoints[-1]  # highest step = last saved

bert_ckpt = find_best_checkpoint("bert-base-uncased")
gpt2_ckpt = find_best_checkpoint("gpt2")
byt5_ckpt = find_best_checkpoint("byt5")
print(f"BERT checkpoint: {bert_ckpt}")
print(f"GPT-2 checkpoint: {gpt2_ckpt}")
print(f"ByT5 checkpoint: {byt5_ckpt}")

In [ ]:
# BERT
model_bert = BertForTokenClassification.from_pretrained(bert_ckpt)
tokenizer_bert = AutoTokenizer.from_pretrained("bert-base-uncased", add_prefix_space=True)

# GPT-2
model_gpt2 = GPT2ForTokenClassification.from_pretrained(gpt2_ckpt)
tokenizer_gpt2 = AutoTokenizer.from_pretrained("gpt2", add_prefix_space=True)
tokenizer_gpt2.pad_token = tokenizer_gpt2.eos_token

# ByT5 — custom class, load weights from checkpoint
from transformers import T5Config
byt5_config = T5Config.from_pretrained("google/byt5-small")
model_byt5 = ByT5ForTokenClassification(byt5_config, num_labels=num_labels)
import torch as _torch
_bin_path = os.path.join(byt5_ckpt, "pytorch_model.bin")
_safetensors_path = os.path.join(byt5_ckpt, "model.safetensors")
if os.path.exists(_bin_path):
    _state = _torch.load(_bin_path, map_location="cpu")
    model_byt5.load_state_dict(_state)
elif os.path.exists(_safetensors_path):
    from safetensors.torch import load_file as _load_safetensors
    _state = _load_safetensors(_safetensors_path)
    model_byt5.load_state_dict(_state)
else:
    raise FileNotFoundError(f"No model weights found in {byt5_ckpt} "
                            "(expected pytorch_model.bin or model.safetensors)")

tokenizer_byt5 = AutoTokenizer.from_pretrained("google/byt5-small")
print("All models loaded.")

## Evaluate on noisy test sets

In [ ]:
def evaluate_on_noisy(model, tokenizer, tokenize_fn, noise_type, model_name,
                       per_device_eval_batch_size=32, tokenize_kwargs=None):
    """Load noisy test set, tokenize, evaluate, return metrics dict."""
    noisy_ds = load_from_disk(os.path.join(DATA_DIR, noise_type))

    kwargs = tokenize_kwargs or {}
    tokenized = noisy_ds["test"].map(
        lambda ex: tokenize_fn(ex, tokenizer, **kwargs),
        batched=True,
        remove_columns=noisy_ds["test"].column_names,
    )

    eval_args = TrainingArguments(
        output_dir="/tmp/eval_tmp",
        per_device_eval_batch_size=per_device_eval_batch_size,
        fp16=(DEVICE == "cuda"),
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=eval_args,
        eval_dataset=tokenized,
        data_collator=DataCollatorForTokenClassification(tokenizer),
        compute_metrics=make_compute_metrics(label_names),
    )
    # Required before evaluate() when not calling train() first
    trainer.callback_handler.on_train_begin(trainer.args, trainer.state, trainer.control)

    metrics = trainer.evaluate()
    return {
        "model": model_name,
        "noise": noise_type,
        "f1": round(metrics["eval_f1"], 4),
        "precision": round(metrics["eval_precision"], 4),
        "recall": round(metrics["eval_recall"], 4),
    }

In [ ]:
results = []

for noise_type in ["ocr", "asr", "social"]:
    print(f"\nEvaluating BERT on {noise_type}...")
    results.append(evaluate_on_noisy(
        model_bert, tokenizer_bert, tokenize_and_align_labels,
        noise_type, "bert-base-uncased"
    ))

    print(f"Evaluating GPT-2 on {noise_type}...")
    results.append(evaluate_on_noisy(
        model_gpt2, tokenizer_gpt2, tokenize_and_align_labels,
        noise_type, "gpt2", per_device_eval_batch_size=16
    ))

    print(f"Evaluating ByT5 on {noise_type}...")
    results.append(evaluate_on_noisy(
        model_byt5, tokenizer_byt5, tokenize_and_align_labels_byt5,
        noise_type, "google/byt5-small", per_device_eval_batch_size=2,
        tokenize_kwargs={"max_length": 256}
    ))

results_df = pd.DataFrame(results)
print("\nDone!")
results_df

## Delta vs Clean Baseline

In [ ]:
# Baseline F1 from notebook 01 results
baseline_f1 = {
    "bert-base-uncased": 0.890836,
    "gpt2": 0.687350,
    "google/byt5-small": 0.862590,
}

results_df["f1_baseline"] = results_df["model"].map(baseline_f1)
results_df["f1_delta"] = (results_df["f1"] - results_df["f1_baseline"]).round(4)
results_df["f1_drop_pct"] = (results_df["f1_delta"] / results_df["f1_baseline"] * 100).round(2)

display_df = results_df[["model", "noise", "f1", "f1_baseline", "f1_delta", "f1_drop_pct"]]
display_df

In [ ]:
os.makedirs(os.path.join(RESULTS_DIR, "tables"), exist_ok=True)
results_df.to_csv(os.path.join(RESULTS_DIR, "tables", "noisy_evaluation.csv"), index=False)
print("Saved results/tables/noisy_evaluation.csv")

In [ ]:
pivot = results_df.pivot(index="model", columns="noise", values="f1_delta")
pivot.index.name = "model"
print("\nF1 delta vs clean baseline (negative = degradation):")
print(pivot.to_string())